# Test pystac_client from rs-client-libraries

The pystac_client.Client class is made to read the database, not to write or update it. We have added methods for that in rs_client.stac.catalog_client. 

We are testing in this notebook that they work as intended.

In [ ]:
# For testing, don't commit
import sys
sys.path.insert(0, "/home/jgaucher/projects/rspy/github/rs-demo/notebooks")
import resources.test_localhost

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *

init_demo()
init_dask_cluster_staging(scale=2)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_THREADS_EOPF=4 docker compose up # ...
display(dask_cluster_staging)

In [ ]:
# Other imports
import json
import pystac
from typing import Callable

In [ ]:
# Create a test collection
col_name = "test_pystac_client"
print(f"Stage items in {col_name!r}")
# NOTE: this uses:
#   - catalog_client.remove_collection
#   - catalog_client.add_collection
#   - catalog_client.get_collection
create_test_collection(col_name)

# Stage auxip products
PRODUCT_COUNT = 3
stage_test_objects(auxip_client, PRODUCT_COUNT, col_name)

In [ ]:
def assert_nominal(product_count) -> list[pystac.item.Item]:
    """Assert that all methods return the expected result"""

    landing = catalog_client.get_landing()
    print("Landing page:")
    print(json.dumps(landing, indent=2))
    assert landing

    col_ids = [col.id for col in catalog_client.get_collections()]
    print(f"Collection IDs: {col_ids}")
    assert catalog_client.full_collection_id(None, col_name, "_") in col_ids

    collection = catalog_client.get_collection(col_name)
    print("Collection:")
    display(collection)
    assert collection

    items = list(catalog_client.get_items(col_name))
    print("Items:")
    display(items)
    assert len(items) == product_count

    item = catalog_client.get_item(col_name, items[0].id)
    print("Item:")
    display(item)
    assert item

    searched = catalog_client.search(collections=[col_name])
    print("Search:")
    display(searched)
    assert len(searched) == product_count

    col_queryables = catalog_client.get_collection_queryables(col_name)
    print("Collection queryables:")
    print(json.dumps(col_queryables, indent=2))
    assert col_queryables

    queryables = catalog_client.get_queryables()
    print("Queryables:")
    print(json.dumps(queryables, indent=2))
    assert queryables

    return items

def assert_missing(missing_col_name: str | None = None):
    """Assert that a collection does not exist"""

    missing_col_name = missing_col_name or col_name

    def assert_exception(func_name: str, callable: Callable):
        """Assert that a function call will raise any Exception"""
        try:
            callable()
        except Exception:
            print(f"{func_name!r} failed as excepted.")
        else:
            assert False, f"{func_name!r} was expected to fail !"

    assert_exception("get_collection", lambda: catalog_client.get_collection(missing_col_name))
    assert_exception("get_items", lambda: catalog_client.get_items(missing_col_name))
    assert_exception("search", lambda: catalog_client.search(collections=[missing_col_name]))
    assert_exception("get_collection_queryables", lambda: catalog_client.get_collection_queryables(missing_col_name))

    col_ids = [col.id for col in catalog_client.get_collections()]
    assert catalog_client.full_collection_id(None, missing_col_name, "_") not in col_ids
    
    # assert_exception("", lambda: )

In [ ]:
# Check nominal case
items = assert_nominal(PRODUCT_COUNT)

In [ ]:
# Check missing collection case
assert_missing("no_such_collection")

In [ ]:
# Add an item
new_item = items[0]
new_item.id = "new_item"
new_item.assets = {}
response = catalog_client.add_item(col_name, new_item)
response.raise_for_status()

# Check that the pystac client returns updated responses
assert_nominal(PRODUCT_COUNT + 1)

In [ ]:
# Remove item and check again
response = catalog_client.remove_item(col_name, new_item.id)
response.raise_for_status()
assert_nominal(PRODUCT_COUNT)

In [ ]:
# Remove the collection and check again
catalog_client.remove_collection(col_name)
assert_missing()

In [ ]:
# tester aussi update item et update collection